In [421]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy import signal

In [422]:
fs = 4

Imported reshape function as it will be needed to make row-wise descriptives

In [423]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'EDA', window_size = 60, stride_size = 15, hertz=8, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

Import tonic and phasic files

In [424]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal.csv")
eda = pd.read_csv("../Data/wesad_eda/filtered_eda_signals.csv")

In [425]:
print(tonic.shape)
print(phasic.shape)
print(eda.shape)

(20892, 30)
(20892, 30)
(20892, 30)


This section is just applying the reshape function and verifying the outputs

In [426]:
eda.head()

,S15_EDA,S15_Label,S7_EDA,S7_Label,S3_EDA,S3_Label,S2_EDA,S2_Label,S14_EDA,S14_Label,...,S4_EDA,S4_Label,S13_EDA,S13_Label,S5_EDA,S5_Label,S9_EDA,S9_Label,S6_EDA,S6_Label
0,0.000000,0.0,0.000000,0.0,0.016044,0.0,0.097471,0.0,0.011914,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.037751,0.0,0.033865,0
1,0.073305,0.0,0.485077,0.0,0.178527,0.0,0.492093,0.0,0.175268,0.0,...,0.031140,0.0,0.366758,0.0,0.065186,0.0,0.144301,0.0,0.199618,0
2,0.116746,0.0,0.854691,0.0,0.319746,0.0,0.769926,0.0,0.278055,0.0,...,0.053885,0.0,0.628746,0.0,0.109112,0.0,0.224244,0.0,0.323051,0
3,0.083819,0.0,0.729083,0.0,0.296441,0.0,0.620082,0.0,0.191334,0.0,...,0.044419,0.0,0.511660,0.0,0.085603,0.0,0.197046,0.0,0.275535,0
4,0.062718,0.0,0.609167,0.0,0.256728,0.0,0.491979,0.0,0.120382,0.0,...,0.035439,0.0,0.421939,0.0,0.063158,0.0,0.172040,0.0,0.231554,0


In [427]:
tonic.head()

,S15_Tonic,S15_Label,S7_Tonic,S7_Label,S3_Tonic,S3_Label,S2_Tonic,S2_Label,S14_Tonic,S14_Label,...,S4_Tonic,S4_Label,S13_Tonic,S13_Label,S5_Tonic,S5_Label,S9_Tonic,S9_Label,S6_Tonic,S6_Label
0,0.035955,0.0,0.079843,0.0,0.045032,0.0,0.205885,0.0,0.080949,0.0,...,0.014317,0.0,0.080199,0.0,0.032143,0.0,0.070445,0.0,0.078180,0
1,0.035899,0.0,0.083086,0.0,0.045773,0.0,0.206719,0.0,0.078571,0.0,...,0.016666,0.0,0.085611,0.0,0.030334,0.0,0.070592,0.0,0.078446,0
2,0.035998,0.0,0.087238,0.0,0.047010,0.0,0.207807,0.0,0.076342,0.0,...,0.019126,0.0,0.091607,0.0,0.028587,0.0,0.071115,0.0,0.079124,0
3,0.036244,0.0,0.092260,0.0,0.048724,0.0,0.209145,0.0,0.074261,0.0,...,0.021695,0.0,0.098161,0.0,0.026907,0.0,0.071998,0.0,0.080196,0
4,0.036632,0.0,0.098116,0.0,0.050892,0.0,0.210730,0.0,0.072328,0.0,...,0.024369,0.0,0.105245,0.0,0.025297,0.0,0.073227,0.0,0.081647,0


In [428]:
tonic_reshaped = reshape_function(tonic, name='Tonic')

In [429]:
tonic_reshaped.shape

(2106, 482)

In [430]:
tonic_reshaped.head()

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_470,sample_471,sample_472,sample_473,sample_474,sample_475,sample_476,sample_477,sample_478,sample_479
0,S10,1.0,0.085307,0.085209,0.085102,0.084986,0.084863,0.084733,0.084597,0.084454,...,0.090827,0.090832,0.090838,0.090843,0.090849,0.090855,0.090861,0.090867,0.090874,0.090880
1,S10,1.0,0.101963,0.101926,0.101864,0.101777,0.101667,0.101535,0.101382,0.101209,...,0.089587,0.089575,0.089564,0.089553,0.089544,0.089537,0.089530,0.089524,0.089520,0.089517
2,S10,1.0,0.090172,0.090178,0.090186,0.090196,0.090208,0.090222,0.090238,0.090255,...,0.090568,0.090578,0.090587,0.090596,0.090604,0.090612,0.090620,0.090627,0.090633,0.090639
3,S10,1.0,0.090579,0.090586,0.090591,0.090595,0.090596,0.090597,0.090595,0.090593,...,0.093480,0.093520,0.093562,0.093604,0.093648,0.093693,0.093739,0.093786,0.093835,0.093885
4,S10,1.0,0.090887,0.090895,0.090902,0.090910,0.090918,0.090926,0.090934,0.090942,...,0.094744,0.094717,0.094689,0.094661,0.094632,0.094602,0.094571,0.094539,0.094506,0.094473


Tonic descriptives row-wise for max and mean

In [431]:
def eda_feature_extraction(row, sampling_rate=fs):
    feature_names = [
        "phasic_peak_count",
        "phasic_peak_rate",
        "phasic_mean_amplitude",
        "phasic_max_amplitude",
        "phasic_sum_amplitude",
        "phasic_mean_rise_time",
        "phasic_std_rise_time",
        "phasic_mean_recovery_time",
        "phasic_mean_onset_time",
        "phasic_mean_offset_time",
        "phasic_shape_symmetry",  # New: Ratio of rise to recovery
        "eda_symp_power"          # New: Spectral power band
    ]

    try:
        # 1. Frequency Domain Extraction (Spectral Power)
        # Calculates power in the typical EDA sympathetic band (0.045 - 0.25 Hz)
        try:
            # Adjust window segment size for Welch's method based on row length
            nperseg = min(len(row), 256) 
            freqs, psd = signal.welch(row, fs=sampling_rate, nperseg=nperseg)
            symp_band = (freqs >= 0.045) & (freqs <= 0.25)
            eda_symp_power = np.trapz(psd[symp_band], freqs[symp_band])
        except Exception:
            eda_symp_power = np.nan

        # 2. Time Domain & Phasic Extraction
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)

        peaks = info["SCR_Peaks"]
        onsets = info["SCR_Onsets"]
        recovery = info["SCR_Recovery"]
        amplitudes = info["SCR_Amplitude"]

        peak_count = len(peaks)
        duration_minutes = (len(row) / sampling_rate) / 60

        if peak_count == 0:
            return pd.Series(
                [0, 0.0] + [np.nan] * 8 + [np.nan, eda_symp_power],
                index=feature_names,
            )

        # Convert sample indices to time durations (in seconds)
        rise_times = (peaks - onsets) / sampling_rate
        recovery_times = (recovery - peaks) / sampling_rate

        mean_rise = np.nanmean(rise_times)
        mean_recovery = np.nanmean(recovery_times)

        # 3. Phasic Shape Morphometrics
        # Ratio of rise time to recovery time (measures morphological asymmetry)
        shape_symmetry = mean_rise / mean_recovery if mean_recovery > 0 else np.nan

        onset_times = onsets / sampling_rate
        offset_times = recovery / sampling_rate 
        peak_rate = peak_count / duration_minutes

        return pd.Series(
            [
                peak_count,
                peak_rate,
                np.nanmean(amplitudes),
                np.nanmax(amplitudes),
                np.nansum(amplitudes),
                mean_rise,
                np.nanstd(rise_times),
                mean_recovery,
                np.nanmean(onset_times),
                np.nanmean(offset_times),
                shape_symmetry,
                eda_symp_power
            ],
            index=feature_names,
        )

    except Exception:
        return pd.Series([np.nan] * len(feature_names), index=feature_names)

In [432]:
# intend to apply this function to phasic reshaped
def rise_time_std(row, sampling_rate = fs):
    try:
        #Uses neurokits peak detection to get the onsets then convert it to a time scale and calculate the standard deviation
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        # Calculate peak count
        peaks = info["SCR_Peaks"]
        peak_count = len(peaks)
        rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
        rise_time_std = np.nanstd(rise_times)
        return pd.Series(
                    [peak_count, rise_time_std],
                    index=["phasic_peak_count", "phasic_std_rise_time"],
                )

    #There is a lot of rows with no peaks at all, need to decide how to handle the NAs where the std depreciates thereof
    except Exception:
        return np.nan

In [433]:
def sig2features(sig, sig_name):
    """
    sig is a signal, like tonic_reshaped
    sig_name is a string, like "tonic"
    #feature_set is a list, like [min, max]
    """
    # only want the signals, no "subject_id" and "label" columns
    sample_cols = [col for col in sig.columns if col.startswith('sample_')]
    sig_descriptives = pd.DataFrame()
    # Bring Labels into descriptives to the database
    sig_descriptives['label'] = sig['label']
    # Bring subject ID over to descriptives
    if 'subject_id' in sig.columns:
        sig_descriptives['subject_id'] = sig['subject_id']
    # Descriptives
    sig_descriptives[f"{sig_name}_mean"] = sig[sample_cols].mean(axis=1)
    sig_descriptives[f"{sig_name}_max"]  = sig[sample_cols].max(axis=1)
    sig_descriptives[f"{sig_name}_std"]  = sig[sample_cols].std(axis=1)
    sig_descriptives[f"{sig_name}_skew"]  = sig[sample_cols].skew(axis=1)
    sig_descriptives[f"{sig_name}_norm1"]  = sig[sample_cols].abs().sum(axis=1)
    sig_descriptives[f"{sig_name}_median"] = sig[sample_cols].median(axis=1)
    sig_descriptives[f"{sig_name}_min"] = sig[sample_cols].min(axis=1)
    sig_descriptives[f"{sig_name}_range"] = sig[sample_cols].max(axis=1) - sig[sample_cols].min(axis=1)
    sig_descriptives[f"{sig_name}_mean_slope"] = (sig[sample_cols].diff(axis=1).mean(axis=1))
    sig_descriptives[f"{sig_name}_kurtosis"] = sig[sample_cols].kurtosis(axis=1)
    sig_descriptives[f"{sig_name}_sampen"] = sig[sample_cols].apply(
    lambda row: nk.entropy_sample(row.dropna())[0] if not row.dropna().empty else np.nan, 
    axis=1)
    sig_descriptives[f"{sig_name}_apen"] = sig[sample_cols].apply(
    lambda row: nk.entropy_approximate(row.dropna())[0] if not row.dropna().empty else np.nan, 
    axis=1)



    if sig_name == 'phasic':
        # Define the full list of features returned by our function
        phasic_cols = [
            "phasic_peak_count",
            "phasic_peak_rate",
            "phasic_mean_amplitude",
            "phasic_max_amplitude",
            "phasic_sum_amplitude",
            "phasic_mean_rise_time",
            "phasic_std_rise_time",
            "phasic_mean_recovery_time",
            "phasic_mean_onset_time",
            "phasic_mean_offset_time",
            "phasic_shape_symmetry",
            "eda_symp_power"
        ]
        
        # Assign the apply output to all 12 columns
        sig_descriptives[phasic_cols] = sig[sample_cols].apply(eda_feature_extraction, axis=1)
        
    return sig_descriptives

In [434]:
tonic_descriptives = sig2features(tonic_reshaped, 'tonic')
tonic_descriptives.head()

,label,subject_id,tonic_mean,tonic_max,tonic_std,tonic_skew,tonic_norm1,tonic_median,tonic_min,tonic_range,tonic_mean_slope,tonic_kurtosis,tonic_sampen,tonic_apen
0,1.0,S10,0.090346,0.101973,0.004471,0.234954,43.366181,0.090381,0.080413,0.021560,1.163615e-05,1.342091,0.013416,0.067586
1,1.0,S10,0.091001,0.101963,0.002209,3.686501,43.680664,0.090467,0.089488,0.012475,-2.598276e-05,13.225404,0.025814,0.025816
2,1.0,S10,0.090388,0.091087,0.000406,-0.272905,43.386399,0.090398,0.089515,0.001572,9.753950e-07,-0.580229,0.120097,0.120154
3,1.0,S10,0.090744,0.093885,0.000900,1.547910,43.557222,0.090595,0.089515,0.004370,6.902364e-06,2.417952,0.041910,0.034193
4,1.0,S10,0.091941,0.096119,0.002163,0.769793,44.131642,0.090951,0.089515,0.006603,7.485093e-06,-0.963250,0.022053,0.021984


Phasic descriptives row-wise for mean, peak counts and rise time standard deviation

In [435]:
# Test example before I implemented on full dataframe the rise time standard deviation descriptive
phasic_reshape = reshape_function(phasic, name='Phasic')
row_example = phasic_reshape.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=fs)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / fs
np.nanstd(rise_times_sec)


0.8385254915624212

In [436]:

phasic_descriptives = sig2features(phasic_reshape, 'phasic')
phasic_descriptives.head()

/tmp/ipykernel_399236/1131894471.py:51: RuntimeWarning: Mean of empty slice
  mean_recovery = np.nanmean(recovery_times)
/tmp/ipykernel_399236/1131894471.py:72: RuntimeWarning: Mean of empty slice
  np.nanmean(offset_times),
/home/sj/.local/lib/python3.12/site-packages/neurokit2/eda/eda_peaks.py:122: RuntimeWarning: All-NaN slice encountered
  valid_peaks = np.logical_and(info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"]))  # pylint: disable=E1111
/tmp/ipykernel_399236/1131894471.py:50: RuntimeWarning: Mean of empty slice
  mean_rise = np.nanmean(rise_times)
/tmp/ipykernel_399236/1131894471.py:51: RuntimeWarning: Mean of empty slice
  mean_recovery = np.nanmean(recovery_times)
/tmp/ipykernel_399236/1131894471.py:65: RuntimeWarning: Mean of empty slice
  np.nanmean(amplitudes),
/tmp/ipykernel_399236/1131894471.py:66: RuntimeWarning: All-NaN slice encountered
  np.nanmax(amplitudes),
/home/sj/.local/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879

,label,subject_id,phasic_mean,phasic_max,phasic_std,phasic_skew,phasic_norm1,phasic_median,phasic_min,phasic_range,...,phasic_mean_amplitude,phasic_max_amplitude,phasic_sum_amplitude,phasic_mean_rise_time,phasic_std_rise_time,phasic_mean_recovery_time,phasic_mean_onset_time,phasic_mean_offset_time,phasic_shape_symmetry,eda_symp_power
0,1.0,S10,0.002559,0.458556,0.026082,14.192955,1.228547,3.013874e-08,4.381018e-09,0.458556,...,0.193435,0.458556,0.773739,1.125000,0.838525,0.583333,9.812500,13.583333,1.928571,NaN
1,1.0,S10,0.000216,0.024459,0.001802,9.980912,0.103619,2.987824e-08,1.336043e-08,0.024459,...,0.012416,0.024459,0.074495,2.166667,1.806624,0.250000,18.916667,23.100000,8.666667,NaN
2,1.0,S10,0.000085,0.024459,0.001234,17.406168,0.040605,3.023632e-08,1.651956e-08,0.024459,...,0.015473,0.024459,0.030947,4.000000,0.750000,0.250000,14.250000,18.500000,16.000000,NaN
3,1.0,S10,0.000216,0.034237,0.002243,12.075679,0.103537,3.093384e-08,1.235907e-08,0.034237,...,0.021629,0.034237,0.064886,5.416667,2.044641,0.250000,97.250000,102.916667,21.666667,NaN
4,1.0,S10,0.000145,0.034237,0.001901,15.345547,0.069437,3.142409e-08,1.235907e-08,0.034237,...,0.021629,0.034237,0.064886,5.416667,2.044641,0.250000,67.250000,72.916667,21.666667,NaN


Create and save file descriptive file

In [437]:
eda_reshaped = reshape_function(eda, name='EDA')
eda_descriptives = sig2features(eda_reshaped, 'eda')
eda_descriptives.head()

,label,subject_id,eda_mean,eda_max,eda_std,eda_skew,eda_norm1,eda_median,eda_min,eda_range,eda_mean_slope,eda_kurtosis,eda_sampen,eda_apen
0,1.0,S10,0.093689,0.158848,0.009054,3.665468,44.970615,0.090649,0.075917,0.082931,6.548994e-06,16.612144,0.094545,0.229623
1,1.0,S10,0.091353,0.105202,0.003022,3.391384,43.849551,0.090616,0.088191,0.017010,-3.189322e-05,11.202429,0.399931,0.527375
2,1.0,S10,0.090500,0.097583,0.000907,2.661236,43.440095,0.090448,0.088669,0.008914,6.757988e-07,14.785969,1.312522,1.186124
3,1.0,S10,0.091040,0.101435,0.001647,2.536093,43.699344,0.090673,0.088669,0.012766,9.640598e-06,9.409743,0.780159,0.827954
4,1.0,S10,0.092134,0.101435,0.002414,0.802010,44.224514,0.091061,0.088669,0.012766,7.051074e-06,-0.237704,0.544766,0.623854


In [438]:
feature_dataframe = pd.concat(
    [eda_descriptives,
     phasic_descriptives.drop(columns=["subject_id", "label"]), 
     tonic_descriptives.drop(columns=["subject_id", "label"]),
     ],
    axis=1,
)
feature_dataframe.head()

,label,subject_id,eda_mean,eda_max,eda_std,eda_skew,eda_norm1,eda_median,eda_min,eda_range,...,tonic_std,tonic_skew,tonic_norm1,tonic_median,tonic_min,tonic_range,tonic_mean_slope,tonic_kurtosis,tonic_sampen,tonic_apen
0,1.0,S10,0.093689,0.158848,0.009054,3.665468,44.970615,0.090649,0.075917,0.082931,...,0.004471,0.234954,43.366181,0.090381,0.080413,0.021560,1.163615e-05,1.342091,0.013416,0.067586
1,1.0,S10,0.091353,0.105202,0.003022,3.391384,43.849551,0.090616,0.088191,0.017010,...,0.002209,3.686501,43.680664,0.090467,0.089488,0.012475,-2.598276e-05,13.225404,0.025814,0.025816
2,1.0,S10,0.090500,0.097583,0.000907,2.661236,43.440095,0.090448,0.088669,0.008914,...,0.000406,-0.272905,43.386399,0.090398,0.089515,0.001572,9.753950e-07,-0.580229,0.120097,0.120154
3,1.0,S10,0.091040,0.101435,0.001647,2.536093,43.699344,0.090673,0.088669,0.012766,...,0.000900,1.547910,43.557222,0.090595,0.089515,0.004370,6.902364e-06,2.417952,0.041910,0.034193
4,1.0,S10,0.092134,0.101435,0.002414,0.802010,44.224514,0.091061,0.088669,0.012766,...,0.002163,0.769793,44.131642,0.090951,0.089515,0.006603,7.485093e-06,-0.963250,0.022053,0.021984


In [439]:
feature_dataframe.shape

(2106, 50)

In [440]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe.csv", index=False)
